# H2O AutoML
#### It loads the dataset, converts it into an H2OFrame, preprocesses categorical features, and then runs H2O's AutoML to automatically train and evaluate multiple models. Finally, it prints the leaderboard of trained models and saves the best-performing model.

##### Install necessary libraries if not already installed Run these in the terminal or command prompt before executing the script


In [ ]:
!pip3 install h2o pandas

In [1]:
import h2o  # Import H2O for machine learning
import pandas as pd  # Import pandas for data manipulation
from h2o.automl import H2OAutoML  # Import H2O's AutoML for automated model training

h2o.init()

Checking whether there is an H2O instance running at http://localhost:54321. connected.
Please download and install the latest version from: https://h2o-release.s3.amazonaws.com/h2o/latest_stable.html


H2O_cluster_uptime:,1 min 28 secs
H2O_cluster_timezone:,America/Chicago
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.6
H2O_cluster_version_age:,6 months and 1 day
H2O_cluster_name:,H2O_from_python_prasadkatkade_dysbjm
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,1.951 Gb
H2O_cluster_total_cores:,8
H2O_cluster_allowed_cores:,8
H2O_cluster_status:,"locked, healthy"


##### Load dataset into a Pandas DataFrame, Convert the Pandas DataFrame into an H2OFrame, required for H2O models

In [2]:
df = pd.read_csv("sales_data.csv") 
df_h2o = h2o.H2OFrame(df)

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


#####  Convert categorical features into "factors" to ensure they are handled correctly by H2O

In [3]:
for col in df_h2o.columns:
    if df_h2o[col].isfactor()[0]:  
        df_h2o[col] = df_h2o[col].asfactor()  

df_h2o["SIM_DATE_year"] = df_h2o["SIM_DATE"].year()
df_h2o["SIM_DATE_month"] = df_h2o["SIM_DATE"].month()
df_h2o["SIM_DATE_day"] = df_h2o["SIM_DATE"].day()

#####  Define the target variable (the value we want to predict), Define predictor variables (all other columns except the target)

In [4]:
target = "QUANTITY"
features = [col for col in df_h2o.columns if col != target]


##### Initialize H2O AutoML with a limit of 10 models and a random seed for reproducibility


In [5]:
aml = H2OAutoML(max_models=10, seed=42)

##### Train the AutoML models using the training dataset


In [6]:
aml.train(x=features, y=target, training_frame=df_h2o)

AutoML progress: |
15:50:08.206: AutoML: XGBoost is not available; skipping it.
15:50:08.207: _train param, Dropping bad and constant columns: [CURRENCY, MATERIAL_TYPE, SIM_DATE_year, SIM_DATE_month, UNIT, SIM_DATE_day, COUNTRY, STORAGE_LOCATION, DISTRIBUTION_CHANNEL, ID]
15:50:08.370: _train param, Dropping bad and constant columns: [CURRENCY, MATERIAL_TYPE, SIM_DATE_year, SIM_DATE_month, UNIT, SIM_DATE_day, COUNTRY, STORAGE_LOCATION, DISTRIBUTION_CHANNEL, ID]

██
15:50:09.744: _train param, Dropping bad and constant columns: [CURRENCY, MATERIAL_TYPE, SIM_DATE_year, SIM_DATE_month, UNIT, SIM_DATE_day, COUNTRY, STORAGE_LOCATION, DISTRIBUTION_CHANNEL, ID]

█
15:50:10.647: _train param, Dropping bad and constant columns: [CURRENCY, MATERIAL_TYPE, SIM_DATE_year, SIM_DATE_month, UNIT, SIM_DATE_day, COUNTRY, STORAGE_LOCATION, DISTRIBUTION_CHANNEL, ID]

█
15:50:11.199: _train param, Dropping bad and constant columns: [CURRENCY, MATERIAL_TYPE, SIM_DATE_year, SIM_DATE_month, UNIT, SIM_DATE_day

Model Details
=============
H2OGradientBoostingEstimator : Gradient Boosting Machine
Model Key: GBM_grid_1_AutoML_2_20250503_155008_model_1


Model Summary: 
    number_of_trees    number_of_internal_trees    model_size_in_bytes    min_depth    max_depth    mean_depth    min_leaves    max_leaves    mean_leaves
--  -----------------  --------------------------  ---------------------  -----------  -----------  ------------  ------------  ------------  -------------
    94                 94                          73681                  6            6            6             13            63            45.3085

ModelMetricsRegression: gbm
** Reported on train data. **

MSE: 0.06413713210405658
RMSE: 0.2532530989031656
MAE: 0.18277527339383268
RMSLE: 0.0023082175439870065
Mean Residual Deviance: 0.06413713210405658

ModelMetricsRegression: gbm
** Reported on cross-validation data. **

MSE: 4.920010957154381
RMSE: 2.218109771213855
MAE: 0.784030760262009
RMSLE: 0.06326396532366887
Mean Residual Deviance: 4.920010957154381

Cross-Validation Metrics Summary: 
                        mean       sd           cv_1_valid    cv_2_valid    cv_3_valid    cv_4_valid    cv_5_valid
----------------------  ---------  -----------  ------------  ------------  ------------  ------------  ------------
aic                     nan        0            nan           nan           nan           nan           nan
loglikelihood           nan        0            nan           nan           nan           nan           nan
mae                     0.784147   0.0512815    0.824744      0.713375      0.746044      0.81197       0.824599
mean_residual_deviance  4.91932    2.77722      9.63134       3.52158       4.45675       4.59235       2.3946
mse                     4.91932    2.77722      9.63134       3.52158       4.45675       4.59235       2.3946
r2                      0.999363   0.000288186  0.998887      0.999483      0.999397      0.999386      0.999662
residual_deviance       4.91932    2.77722      9.63134       3.52158       4.45675       4.59235       2.3946
rmse                    2.15631    0.580567     3.10344       1.87659       2.1111        2.14298       1.54745
rmsle                   0.0494712  0.0440299    0.112647      0.0114833     0.0565556     0.0629096     0.00376036

Scoring History: 
    timestamp            duration    number_of_trees    training_rmse    training_mae    training_deviance
--  -------------------  ----------  -----------------  ---------------  --------------  -------------------
    2025-05-03 15:50:14  0.352 sec   0                  86.5757          67.4914         7495.35
    2025-05-03 15:50:14  0.361 sec   5                  51.2498          39.9288         2626.54
    2025-05-03 15:50:14  0.370 sec   10                 30.3719          23.6246         922.453
    2025-05-03 15:50:14  0.379 sec   15                 18.0395          13.9917         325.425
    2025-05-03 15:50:14  0.387 sec   20                 10.6494          8.26688         113.409
    2025-05-03 15:50:14  0.395 sec   25                 6.33006          4.89161         40.0696
    2025-05-03 15:50:14  0.403 sec   30                 3.798            2.9061          14.4248
    2025-05-03 15:50:14  0.411 sec   35                 2.32054          1.74951         5.38491
    2025-05-03 15:50:14  0.419 sec   40                 1.45125          1.06804         2.10613
    2025-05-03 15:50:15  0.428 sec   45                 0.941667         0.672752        0.886736
    2025-05-03 15:50:15  0.436 sec   50                 0.676273         0.450904        0.457345
    2025-05-03 15:50:15  0.445 sec   55                 0.518046         0.329756        0.268372
    2025-05-03 15:50:15  0.453 sec   60                 0.425007         0.26987         0.180631
    2025-05-03 15:50:15  0.461 sec   65                 0.371251         0.239221        0.137827
    2025-05-03 15:50:15  0.467 sec   70                 0.336951         0.223355        0.11353

##### Print the leaderboard displaying all trained models ranked by performance, and save the best model

In [7]:
print(aml.leaderboard)
model_path = h2o.save_model(aml.leader, path="./best_quantity_model", force=True)
print("Model saved at:", model_path)

model_id                                                    rmse       mse       mae      rmsle    mean_residual_deviance
GBM_grid_1_AutoML_2_20250503_155008_model_1              2.21811   4.92001  0.784031  0.063264                    4.92001
StackedEnsemble_BestOfFamily_1_AutoML_2_20250503_155008  2.38706   5.69808  1.12168   0.0645051                   5.69808
StackedEnsemble_AllModels_1_AutoML_2_20250503_155008     2.47318   6.1166   1.07502   0.0687782                   6.1166
GBM_5_AutoML_2_20250503_155008                           3.80674  14.4913   1.0837    0.0806265                  14.4913
GBM_3_AutoML_2_20250503_155008                           4.02787  16.2237   1.35265   0.0857952                  16.2237
GBM_4_AutoML_2_20250503_155008                           4.04035  16.3244   1.31907   0.0852126                  16.3244
GBM_2_AutoML_2_20250503_155008                           4.2837   18.3501   1.55177   0.0841666                  18.3501
XRT_1_AutoML_2_20250503_15500